# CreditMetrics-Style Portfolio Credit Risk via Rating Transition Matrices


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import numpy as np

Data Preparation - Ratings, transition matrix, current and forward value, Portfolio Specifics

In [ ]:
## Data 
Ratings = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC", "D"]

# Transition Matrix (rows: starting rating, columns: end rating - 7*8 matrix)
prob_transition = np.array([
[0.91115, 0.08179, 0.00607, 0.00072, 0.00024, 0.00003, 0.00000, 0.00000],
[0.00844, 0.89626, 0.08954, 0.00437, 0.00064, 0.00036, 0.00018, 0.00021],
[0.00055, 0.02595, 0.91138, 0.05509, 0.00499, 0.00107, 0.00045, 0.00052],
[0.00031, 0.00147, 0.04289, 0.90584, 0.03898, 0.00708, 0.00175, 0.00168],
[0.00007, 0.00044, 0.00446, 0.06741, 0.83274, 0.07667, 0.00895, 0.00926],
[0.00008, 0.00031, 0.00150, 0.00490, 0.05373, 0.82531, 0.07894, 0.03523],
[0.00000, 0.00015, 0.00023, 0.00091, 0.00388, 0.07630, 0.83035, 0.08818]
])

## Sum of each row should be 1
row_sums = prob_transition.sum(axis=1)
print("Row sums (should be 1):", row_sums)


V0 = {"AAA":99.40, "AA":98.39, "A":97.22, "BBB":92.79, "BB":90.11, "B":86.60, "CCC":77.16}
V1 = {"AAA":99.50, "AA":98.51, "A":97.53, "BBB":92.77, "BB":90.48, "B":88.25, "CCC":77.88, "D":60.00}

portfolios = {
    "Portfolio I": [("AAA", 0.60), ("AA", 0.30), ("BBB", 0.10)],
    "Portfolio II": [("BB", 0.60), ("B", 0.35), ("CCC", 0.05)]
}



Building Threshold/bins

In [ ]:
# Ratings order for bins: D..AAA
rev_ratings = Ratings[::-1]  # ["D","CCC","B","BB","BBB","A","AA","AAA"]

# Build cutpoints for every initial rating row

thresholds = {}

for i, start_rating in enumerate(Ratings[:-1]):  # exclude "D" as a starting state
    p_row = prob_transition[i]         
    p_rev = p_row[::-1]  # Reverse to match rev_ratings order       
    cdf_rev = np.cumsum(p_rev)        
    cuts = stats.norm.ppf(np.clip(cdf_rev[:-1], 1e-12, 1-1e-12))
    thresholds[start_rating] = cuts

print("Built thresholds for:", list(thresholds.keys()))
print("Example cutpoints for CCC:", thresholds["CCC"])



In [ ]:
for G, cuts in thresholds.items():
    assert np.all(np.diff(cuts) >= -1e-12), f"Non-monotone cuts for {G}"
print("All cutpoints are monotone increasing.")


Mappting latent variable X to end rating

In [ ]:
def map_x_to_rating(x, cuts, rev_ratings):
    k = np.searchsorted(cuts, x, side="right")  # returns an integer 0..7
    return rev_ratings[k]


Model Validation - Default threshold and convergence check

In [ ]:
##Model Validation

# --- Default threshold check for BBB ---
idx_BBB = Ratings.index("BBB")
idx_D   = Ratings.index("D")

p_default_BBB = prob_transition[idx_BBB, idx_D]         # P(BBB -> D)
z_score  = stats.norm.ppf(p_default_BBB)                # Φ^{-1}(p)
z_score_from_bins = thresholds["BBB"][0]                # first cutpoint

print("P(BBB->D):", p_default_BBB)
print("Z-score:", z_score)
print("Thresholds['BBB'][0]:", z_score_from_bins)
print("Difference:", z_score_from_bins - z_score)




In [ ]:
def portfolio_II_var995(rho, N, seed, PV0=1500.0):
    rng = np.random.default_rng(seed)

    exposures = [("BB", 0.60*PV0), ("B", 0.35*PV0), ("CCC", 0.05*PV0)]

    # one systematic factor per scenario
    Y = rng.standard_normal(N)

    PV1 = np.zeros(N)

    for start_rating, mv0 in exposures:
        eps = rng.standard_normal(N)  # issuer-specific shock (Q1: 1 issuer per bucket)
        X = np.sqrt(rho)*Y + np.sqrt(1-rho)*eps

        cuts = thresholds[start_rating]
        bin_idx = np.searchsorted(cuts, X, side="right")  # 0..7
        end = np.array(rev_ratings)[bin_idx]              # D..AAA

        v0 = V0[start_rating]
        v1 = np.array([V1[r] for r in end])

        PV1 += mv0 * (v1 / v0)

    loss = PV0 - PV1
    return float(np.quantile(loss, 0.995))



In [ ]:
def three_seed_var_range(rho, N, seeds=(42,43,44)):
    vars_ = [portfolio_II_var995(rho, N, s) for s in seeds]
    return vars_, (max(vars_) - min(vars_)), float(np.mean(vars_))



In [ ]:
rho = 0.33
seeds = (42, 43, 44)
target_rel = 0.01  # 1%

N_grid = [50_000, 100_000, 200_000, 300_000, 500_000, 800_000, 1_000_000]

for N in N_grid:
    vars_, rng_width, mean_var = three_seed_var_range(rho, N, seeds=seeds)
    rel = rng_width / mean_var

    print(f"N={N}  VaR99.5={vars_}  Range={rng_width:.4f}  Rel%={100*rel:.2f}")

    if rel <= target_rel:
        final_N = N
        final_vars = vars_
        final_range = rng_width
        final_rel = rel
        break



Calculations of Question 1 - Expected Portfolio Value and Risk metrics for different asset correlation (Single Issuer)

In [ ]:
## Expected Portfolio and Risk Metrics(Var and ES)

PV0 = 1500.0
N = final_N if "final_N" in globals() else 300_000 
seed = 42

rhos = {"0%": 0.0, "33%": 0.33, "66%": 0.66, "100%": 1.0}
portfolios = {
    "Portfolio I": [("AAA", 0.60), ("AA", 0.30), ("BBB", 0.10)],
    "Portfolio II": [("BB", 0.60), ("B", 0.35), ("CCC", 0.05)]
}


def simulate_portfolio_values(portfolio, rho, N, seed, PV0):
    rng = np.random.default_rng(seed)
    Y = rng.standard_normal(N)          # systematic factor (one per scenario)
    PV1 = np.zeros(N)

    for start_rating, w in portfolio:
        mv0 = w * PV0
        eps = rng.standard_normal(N)    # idiosyncratic shock (one issuer per bucket)
        X = np.sqrt(rho) * Y + np.sqrt(1 - rho) * eps

        cuts = thresholds[start_rating]
        bin_idx = np.searchsorted(cuts, X, side="right")    
        end = np.array(rev_ratings)[bin_idx]                 

        v0 = V0[start_rating]
        v1 = np.array([V1[r] for r in end])
        PV1 += mv0 * (v1 / v0)

    return PV1

def risk_metrics_from_PV(PV1, PV0):
    loss = PV0 - PV1
    EV = float(PV1.mean())

    VaR90 = float(np.quantile(loss, 0.90))
    VaR995 = float(np.quantile(loss, 0.995))

    ES90 = float(loss[loss >= VaR90].mean())
    ES995 = float(loss[loss >= VaR995].mean())

    return EV, VaR90, VaR995, ES90, ES995

# ---- Run grid & build table ----
rows = []
for pname, portfolio in portfolios.items():
    for rho_label, rho in rhos.items():
        PV1 = simulate_portfolio_values(portfolio, rho, N, seed, PV0)
        EV, VaR90, VaR995, ES90, ES995 = risk_metrics_from_PV(PV1, PV0)
        rows.append({
            "Portfolio": pname,
            "Rho": rho_label,
            "Expected Value": EV,
            "90% VaR": VaR90,
            "99.5% VaR": VaR995,
            "90% ES": ES90,
            "99.5% ES": ES995
        })

df_results = pd.DataFrame(rows).sort_values(["Portfolio", "Rho"]).reset_index(drop=True)
rho_order = ["0%", "33%", "66%", "100%"]
df_results["Rho"] = pd.Categorical(df_results["Rho"], categories=rho_order, ordered=True)
df_results = df_results.sort_values(["Portfolio", "Rho"]).reset_index(drop=True)

# ---- Display (tabular output) ----
pd.set_option("display.float_format", "{:.2f}".format)
print(df_results)


In [ ]:
loss = PV0 - PV1
print("P(gain) =", np.mean(loss < 0))
print("min loss, max loss:", loss.min(), loss.max())


Visual Plots for Q1

In [ ]:
def plot_loss_distributions(ax, portfolio, title, rho_list, rho_labels):
    for rho, lab in zip(rho_list, rho_labels):
        PV1 = simulate_portfolio_values(portfolio, rho, N, seed, PV0)
        loss = PV0 - PV1
        ax.hist(loss, bins=60, density=True, alpha=0.4, label=f"ρ={lab}")

    ax.set_title(title)
    ax.set_xlabel("Loss")
    ax.grid(True)

rho_list   = [0.0, 0.33, 0.66, 1.0]
rho_labels = ["0%", "33%", "66%", "100%"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

plot_loss_distributions(
    axes[0],
    portfolios["Portfolio I"],
    "Portfolio I",
    rho_list,
    rho_labels
)

plot_loss_distributions(
    axes[1],
    portfolios["Portfolio II"],
    "Portfolio II",
    rho_list,
    rho_labels
)

axes[0].set_ylabel("Density")
axes[0].legend()

plt.suptitle("Loss Distributions vs Correlation (Single Issuer per Rating)")
plt.show()
   


Question 2 - 100 issuers per bucket

In [ ]:
def simulate_portfolio_values_Q2(portfolio, rho, N, seed, PV0, issuers_per_bucket=100):
    rng = np.random.default_rng(seed)
    Y = rng.standard_normal(N) 
    PV1 = np.zeros(N)

    for start_rating, w in portfolio:
        mv0 = w * PV0

       
        eps = rng.standard_normal((N, issuers_per_bucket))

      
        X = np.sqrt(rho) * Y[:, None] + np.sqrt(1 - rho) * eps

       
        cuts = thresholds[start_rating]
        bin_idx = np.searchsorted(cuts, X, side="right")      
        end = np.array(rev_ratings)[bin_idx]                  

       
        v0 = V0[start_rating]
        v1 = np.vectorize(V1.get)(end)                        

        bucket_value_factor = (v1 / v0).mean(axis=1)          
        PV1 += mv0 * bucket_value_factor

    return PV1

results_Q2 = []
PV0 = 1500.0
seed = 42
N = final_N if "final_N" in globals() else 300_000

rhos = {"0%": 0.0, "33%": 0.33, "66%": 0.66, "100%": 1.0}

for pname, portfolio in portfolios.items():
    for rho_label, rho in rhos.items():
        PV1 = simulate_portfolio_values_Q2(portfolio, rho, N, seed, PV0, issuers_per_bucket=100)
        EV, VaR90, VaR995, ES90, ES995 = risk_metrics_from_PV(PV1, PV0)

        results_Q2.append({
            "Portfolio": pname,
            "Rho": rho_label,
            "Expected Value": EV,
            "90% VaR": VaR90,
            "99.5% VaR": VaR995,
            "90% ES": ES90,
            "99.5% ES": ES995
        })

df_Q2 = pd.DataFrame(results_Q2).sort_values(["Portfolio", "Rho"]).reset_index(drop=True)
rho_order = ["0%", "33%", "66%", "100%"]
df_Q2["Rho"] = pd.Categorical(df_Q2["Rho"], categories=rho_order, ordered=True)
df_Q2 = df_Q2.sort_values(["Portfolio","Rho"]).reset_index(drop=True)
pd.set_option("display.float_format", "{:.2f}".format)
print(df_Q2)



Visual Plots for Q2

In [ ]:
def plot_loss_distributions(ax, simulator, portfolio, title, rho_list, rho_labels):
    for rho, lab in zip(rho_list, rho_labels):
        PV1 = simulator(portfolio, rho, N, seed, PV0)
        ax.hist(PV0 - PV1, bins=60, density=True, alpha=0.4, label=f"ρ={lab}")
    ax.set_title(title)
    ax.set_xlabel("Loss")
    ax.grid(True)

rho_list   = [0.0, 0.33, 0.66, 1.0]
rho_labels = ["0%", "33%", "66%", "100%"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

plot_loss_distributions(
    axes[0],
    simulate_portfolio_values_Q2,       
    portfolios["Portfolio I"],
    "Portfolio I (Q2: 100 issuers per rating)",
    rho_list,
    rho_labels
)

plot_loss_distributions(
    axes[1],
    simulate_portfolio_values_Q2,       
    portfolios["Portfolio II"],
    "Portfolio II (Q2: 100 issuers per rating)",
    rho_list,
    rho_labels
)

axes[0].set_ylabel("Density")
axes[0].legend()

plt.suptitle("Loss Distributions vs Correlation (Q2)")
plt.show()


Question 3.3 - Tail Fatness

In [ ]:

df = df_Q2 

tmp = df[df["Portfolio"].eq("Portfolio II")].copy()
tmp["ES/VaR (99.5%)"] = tmp["99.5% ES"] / tmp["99.5% VaR"]

print(tmp[["Portfolio", "Rho", "99.5% VaR", "99.5% ES", "ES/VaR (99.5%)"]])
